# Modelo de PD de originación – María Isabel Aristizabal Alzate
Objetivo: construir y validar un modelo de probabilidad de incumplimiento
para solicitantes sin historial en buró, usando datos alternativos.

## 1. Carga de datos

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RAW = Path("../data/raw")
pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")

In [2]:
sol_raw = pd.read_csv(RAW / "dataset_candidato.csv", sep=";", dtype=str, keep_default_na=False)
car_raw = pd.read_csv(RAW / "cartera_variable_respuesta_candidato.csv", dtype=str, keep_default_na=False)
print("Solicitudes:", sol_raw.shape, "| Cartera:", car_raw.shape)

Solicitudes: (4000, 19) | Cartera: (18169, 5)


In [3]:
sol_raw.head()

,application_id,application_date,age,sexo,monthly_income_declared,app_tenure_months,avg_monthly_topups,num_transactions_last_90d,pct_p2p_transfers,device_price_tier,has_bureau_record,bureau_score,utility_payment_ontime_rate,telco_payment_ontime_rate,digital_engagement_score,monto_bono_referidos,requested_amount,product_type,disbursed_flag
0,CAND-02052,1/01/24,22,M,1149897.0,7,14838.0,2,0.712,3,FALSE,,0.524,0.728,52.7,0.0,1160359.0,compra_cartera,TRUE
1,CAND-00752,1/01/24,33,M,616023.0,11,51171.0,3,0.288,3,FALSE,,0.822,0.938,50.3,0.0,273514.0,libre_inversion,TRUE
2,CAND-00542,1/01/24,43,M,3064570.0,7,31961.0,6,0.36,1,FALSE,,0.547,0.915,81.2,0.0,283447.0,libre_inversion,TRUE
3,CAND-01204,1/01/24,34,H,1817355.0,7,43710.0,3,0.515,2,FALSE,,0.501,0.922,59.6,0.0,741165.0,libre_inversion,TRUE
4,CAND-02825,1/01/24,30,M,879261.0,5,37064.0,2,0.544,4,TRUE,530.0,0.697,0.738,19.5,0.0,167681.0,avance_efectivo,TRUE


In [4]:
car_raw.head()

,application_id,fecha_apertura,fecha_corte,altura_mora,saldo_capital
0,CAND-00001,2024-03-20,2024-04-19,0,428989.0
1,CAND-00001,2024-03-20,2024-05-19,0,386445.0
2,CAND-00001,2024-03-20,2024-06-18,0,347446.0
3,CAND-00001,2024-03-20,2024-07-18,0,314018.0
4,CAND-00002,2024-08-22,2024-09-21,0,784934.0


In [5]:
sol = sol_raw.replace("", np.nan)
num_cols = ["age","monthly_income_declared","app_tenure_months","avg_monthly_topups",
            "num_transactions_last_90d","pct_p2p_transfers","device_price_tier","bureau_score",
            "utility_payment_ontime_rate","telco_payment_ontime_rate","digital_engagement_score",
            "monto_bono_referidos","requested_amount"]
pd.DataFrame({
    "no_numericos": {c: int((pd.to_numeric(sol[c], errors="coerce").isna() & sol[c].notna()).sum()) for c in num_cols},
    "nulos": sol[num_cols].isna().sum(),
})

,no_numericos,nulos
age,0,0
monthly_income_declared,49,0
app_tenure_months,0,0
avg_monthly_topups,0,0
num_transactions_last_90d,0,0
pct_p2p_transfers,0,0
device_price_tier,0,0
bureau_score,0,2191
utility_payment_ontime_rate,0,255
telco_payment_ontime_rate,0,241


In [6]:
print("IDs duplicados:", sol["application_id"].duplicated().sum())
for c in ["sexo", "product_type", "has_bureau_record", "disbursed_flag", "device_price_tier"]:
    print(c, sol[c].value_counts(dropna=False).to_dict())

IDs duplicados: 0
sexo {'H': 1984, 'M': 1853, 'NB': 163}
product_type {'libre_inversion': 2192, 'avance_efectivo': 1004, 'compra_cartera': 804}
has_bureau_record {'FALSE': 2191, 'TRUE': 1809}
disbursed_flag {'TRUE': 3930, 'FALSE': 70}
device_price_tier {'1': 1236, '2': 1087, '3': 848, '4': 554, '5': 275}


In [7]:
sol.loc[pd.to_numeric(sol["monthly_income_declared"], errors="coerce").isna(),
        ["application_id", "monthly_income_declared"]].head(10)

,application_id,monthly_income_declared
163,CAND-02580,13.760.566.878.899.200
178,CAND-00518,13.777.722.536.049.300
243,CAND-01908,7.071.149.473.249.980
296,CAND-02728,14.990.568.150.977.100
316,CAND-00750,24.123.345.785.788.900
469,CAND-01013,1.924.125.405.490.580
706,CAND-00506,2.686.018.256.156.650
744,CAND-00272,10.393.513.001.866.500
859,CAND-01997,961.705.411.118.913
930,CAND-02771,9.475.369.019.294.410


### Hallazgos de la carga inicial

- **Formato de archivos:** el dataset de solicitudes viene separado por `;` y el de cartera
  por `,`. Se leyeron ambos como texto (`dtype=str`) para no imponer tipos y poder
  detectar anomalías de formato antes de convertir.
- **Tamaño:** 4.000 solicitudes (19 variables) y 18.169 registros de cartera
  (panel mensual: varios cortes por crédito).
- **Llave:** `application_id` es única en solicitudes (0 duplicados).
- **Ingreso con formato corrupto:** 49 registros de `monthly_income_declared` (1,2 %) no
  son numéricos, p. ej. `13.760.566.878.899.200`. Parece un número decimal exportado con
  separadores de miles locales donde se perdió la coma decimal
  (≈ 1.376.056,68). No se eliminan: en el siguiente paso se intenta recuperarlos.
- **Nulos:**
  - `bureau_score` tiene 2.191 nulos, exactamente el mismo número de solicitudes con
    `has_bureau_record = FALSE` → es un **nulo estructural** (clientes thin-file),
    no un dato faltante.
  - `utility_payment_ontime_rate` (255) y `telco_payment_ontime_rate` (241) tienen
    ~6 % de nulos → faltantes reales de datos alternativos.
- **Desembolso:** 70 solicitudes (1,75 %) no se desembolsaron; no deberían tener
  registro en cartera ni desempeño observable.
- **Categóricas:** `sexo` (H 1.984, M 1.853, NB 163), `product_type` con 3 niveles,
  `device_price_tier` de 1 a 5. Sin categorías inesperadas.